In [39]:
import pandas as pd
import arviz as az
from Functions import *

In [42]:
data=pd.read_excel(r"../Data_and_Preprocessing/before_season_correction.xlsx")
device='cuda'
train=data.loc[data.label==0,:,]

In [10]:
data_pm=pd.read_excel(r"../Data_and_Preprocessing/pm25_BC_corrected.xlsx")
train_pm25=data_pm.loc[data_pm.label==0,:,]
device="cuda"

In [11]:
train=make_periodic(train,"corrected_week",device) 
train_pm25=make_periodic(train_pm25,"corrected_week",device) 


/home/michaelf/Seasonality_pyro_rewrite/Seasonality_functions/Functions.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:,"theta"]=data.loc[:,variable]/data.loc[:,variable].max()*2*np.pi
/home/michaelf/Seasonality_pyro_rewrite/Seasonality_functions/Functions.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:,"cos_theta"]=np.cos(data.loc[:,"theta"])
/home/michaelf/Seasonality_pyro_rewrite/Seasonality_functions/Functions.py:18: SettingWithCopyWarning: 
A value is trying to be set on

In [15]:

X_flow,y_flow=make_Xy(train_pm25,["cos_theta","sin_theta"],"flow",device) 
nuts_kernel_flow,gpr_flow=model(X_flow,y_flow,device) 
pyro_data_flow,gpr_flow=train_model(X_flow,nuts_kernel_flow,gpr_flow,device) 
torch.save(gpr_flow,"models/flow_seasonality");

Sample: 100%|██████████████████████████████████████| 14000/14000 [09:45, 23.92it/s, step size=7.17e-01, acc. prob=0.923]
/data/michaelf/miniconda3/lib/python3.10/site-packages/arviz/data/io_pyro.py:158: UserWarning: Could not get vectorized trace, log_likelihood group will be omitted. Check your model vectorization or set log_likelihood=False
  warnings.warn(


In [17]:
X_sam,y_sam=make_Xy(train,["cos_theta","sin_theta"],"sample_time",device) 
nuts_kernel_samp,gpr_samp=model(X_sam,y_sam,device) 
sampling_pyro,gpr_samp=train_model(X_sam,nuts_kernel_samp,gpr_samp,device) 
torch.save(gpr_samp,"models/sampling_seasonality");

Warmup:  13%|████▋                                | 1757/14000 [2:14:48,  4.60s/it, step size=1.32e+00, acc. prob=0.791]
/data/michaelf/miniconda3/lib/python3.10/site-packages/arviz/data/io_pyro.py:158: UserWarning: Could not get vectorized trace, log_likelihood group will be omitted. Check your model vectorization or set log_likelihood=False
  warnings.warn(


In [24]:
X_pass,y_pass=make_Xy(train_pm25,["cos_theta","sin_theta"],"passive",device) 
nuts_kernel_pass,gpr_pass=model(X_pass,y_pass,device) 
pass_pyro,gpr_pass,=train_model(X_pass,nuts_kernel_pass,gpr_pass,device) 
torch.save(gpr_pass,"models/pass_seasonality");

Sample: 100%|██████████████████████████████████████| 14000/14000 [09:14, 25.24it/s, step size=7.96e-01, acc. prob=0.905]
/data/michaelf/miniconda3/lib/python3.10/site-packages/arviz/data/io_pyro.py:158: UserWarning: Could not get vectorized trace, log_likelihood group will be omitted. Check your model vectorization or set log_likelihood=False
  warnings.warn(
